# Topic 5: Hash Maps & Sets

**Goal**: Understand how hash maps work under the hood, and master the patterns that use them.  
**Time**: ~4-5 hours  
**Prereqs**: Topics 0-2

---

## Why Hash Maps?

Hash maps are the **single most useful data structure** in programming. They give you O(1) average-time lookups, inserts, and deletes. Any time you need to quickly look something up, count something, or group things — hash maps.

| Operation | List | Hash Map (dict) |
|---|---|---|
| Look up by key | O(n) scan | O(1) instant |
| Check existence | O(n) scan | O(1) instant |
| Insert | O(1) append | O(1) |
| Delete by key | O(n) find + shift | O(1) |

---

## How Hashing Works (The Big Picture)

```
"apple"  ──hash function──→ 3    → buckets[3] = ("apple", 5)
"banana" ─hash function──→ 7    → buckets[7] = ("banana", 2)
"cherry" ─hash function──→ 3    → COLLISION! Chain: ("apple",5)→("cherry",8)

The hash function converts any key into a bucket index.
Most of the time: 1 key per bucket = O(1)
Occasionally: collision = small chain = still ~O(1) on average
```

## Under the Hood: The Library Analogy

Imagine a library with **10 shelves**. You need to find a book.

**Without hashing (a list):**
```
You walk shelf by shelf, scanning every book title.
Shelf 0: "Zebra"  ← nope
Shelf 1: "Mango"  ← nope
Shelf 2: "Apple"  ← FOUND! But you checked 3 shelves.
Worst case: check ALL shelves → O(n)
```

**With hashing (a hash map):**
```
The librarian has a FORMULA:
  shelf_number = sum_of_letters(title) % 10

"Apple" → sum of letters = 530 → 530 % 10 = 0 → Go to shelf 0!
One calculation, one lookup → O(1)
```

### The 3-step process:

```
Step 1: Hash Function      key ──→ big integer
        "apple" ──hash()──→ 3429873210487

Step 2: Modulo             big integer ──→ bucket index
        3429873210487 % 10 ──→ 7

Step 3: Store/Retrieve     go directly to that bucket
        buckets[7] = ("apple", 5)
```

### What about collisions?

```
Sometimes two keys hash to the SAME bucket:

"cat"  → hash() % 8 = 3
"dog"  → hash() % 8 = 3   ← same bucket!

Solution: CHAINING — each bucket holds a linked list:

buckets[0]: []
buckets[1]: []
buckets[2]: []
buckets[3]: [("cat", 1) → ("dog", 2)]   ← chain of 2
buckets[4]: []
...

With a good hash function, chains stay short → still ~O(1)
```

In [ ]:
class BasicHashMap:
    """A hash map built from scratch using chaining for collision resolution."""

    def __init__(self, size=8):
        self.size = size
        self.buckets = [[] for _ in range(size)]
        self.count = 0

    def _hash(self, key):
        return hash(key) % self.size

    def put(self, key, value):
        idx = self._hash(key)
        bucket = self.buckets[idx]

        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket[i] = (key, value)
                print(f"  Updated: buckets[{idx}] key='{key}' → {value}")
                return

        bucket.append((key, value))
        self.count += 1
        print(f"  Inserted: buckets[{idx}] ← ('{key}', {value})")

    def get(self, key):
        idx = self._hash(key)
        for k, v in self.buckets[idx]:
            if k == key:
                return v
        raise KeyError(key)

    def remove(self, key):
        idx = self._hash(key)
        bucket = self.buckets[idx]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket.pop(i)
                self.count -= 1
                print(f"  Removed: '{key}' from buckets[{idx}]")
                return
        raise KeyError(key)

    def __repr__(self):
        lines = [f"BasicHashMap (size={self.size}, items={self.count})"]
        for i, bucket in enumerate(self.buckets):
            if bucket:
                chain = " → ".join(f"('{k}',{v})" for k, v in bucket)
                lines.append(f"  [{i}]: {chain}")
            else:
                lines.append(f"  [{i}]: empty")
        return "\n".join(lines)


hm = BasicHashMap(size=4)

print("=== Inserting items ===")
hm.put("apple", 5)
hm.put("banana", 2)
hm.put("cherry", 8)
hm.put("date", 3)
hm.put("elderberry", 7)

print("\n=== Current state ===")
print(hm)

print("\n=== Lookups ===")
print(f"  get('apple')  = {hm.get('apple')}")
print(f"  get('cherry') = {hm.get('cherry')}")

print("\n=== Update 'apple' to 99 ===")
hm.put("apple", 99)

print("\n=== Remove 'banana' ===")
hm.remove("banana")

print("\n=== Final state ===")
print(hm)

Now that you see how it works under the hood, let's use Python's built-in `dict` and `set` (which are highly optimized hash maps) to solve problems.

- `dict` → hash map with key-value pairs
- `set` → hash map with only keys (no values), optimized for membership testing

```
dict:  { key → value }    "Is this key here? What's its value?"
set:   { key }            "Is this key here? Yes/No."
```

---

## Core Problem 1: Two Sum (The Classic)

**THE** most famous interview problem. Given an array and a target, find two numbers that add up to the target. Return their indices.

### The Hash Map Insight

For each number, ask: "Is my **complement** (target - num) already in my map?"

```
nums = [2, 7, 11, 15], target = 9

Step 1: num=2, need 9-2=7. Map: {}.       7 not found. Store {2: idx0}
Step 2: num=7, need 9-7=2. Map: {2: 0}.   2 FOUND at idx 0! Return [0, 1]

         ┌─────────────────────────┐
         │  For each num:          │
         │  complement = target-num│
         │  if complement in map:  │
         │      return answer!     │
         │  else:                  │
         │      map[num] = index   │
         └─────────────────────────┘
```

One pass through the array. **O(n) time, O(n) space.**

In [ ]:
def two_sum(nums, target):
    seen = {}
    for i, num in enumerate(nums):
        complement = target - num
        print(f"  i={i}, num={num}, need={complement}, map={seen}")
        if complement in seen:
            print(f"  >>> FOUND! {complement} is at index {seen[complement]}")
            return [seen[complement], i]
        seen[num] = i
    return []


print("=== Two Sum ===")
print("nums = [2, 7, 11, 15], target = 9")
result = two_sum([2, 7, 11, 15], 9)
print(f"Result: {result}\n")

print("nums = [3, 2, 4], target = 6")
result = two_sum([3, 2, 4], 6)
print(f"Result: {result}")

---

## Core Problem 2: Frequency Counting

Counting how often each element appears — the **bread and butter** of hash maps.

```
"banana"

Step through each character:
  'b' → freq = {'b': 1}
  'a' → freq = {'b': 1, 'a': 1}
  'n' → freq = {'b': 1, 'a': 1, 'n': 1}
  'a' → freq = {'b': 1, 'a': 2, 'n': 1}
  'n' → freq = {'b': 1, 'a': 2, 'n': 2}
  'a' → freq = {'b': 1, 'a': 3, 'n': 2}

Pattern:
  for item in collection:
      freq[item] = freq.get(item, 0) + 1
```

In [ ]:
from collections import Counter


# --- Manual counting with dict ---
print("=== Manual Frequency Count ===")
word = "mississippi"
freq = {}
for ch in word:
    freq[ch] = freq.get(ch, 0) + 1
    print(f"  '{ch}' → {dict(freq)}")

print(f"\nFinal: {freq}")

# --- Same thing with Counter (one line) ---
print("\n=== Using Counter ===")
freq2 = Counter(word)
print(f"Counter: {freq2}")

print(f"\nMost common letter: {freq2.most_common(1)}")
print(f"Least common letter: {freq2.most_common()[-1]}")

# --- Top K Frequent Elements ---
print("\n=== Top K Frequent Elements ===")
nums = [1, 1, 1, 2, 2, 3, 3, 3, 3, 4]
k = 2
counts = Counter(nums)
print(f"nums = {nums}")
print(f"Counts: {counts}")
top_k = [num for num, _ in counts.most_common(k)]
print(f"Top {k} frequent: {top_k}")

---

## Core Problem 3: First Non-Repeating Character

Given a string, find the first character that appears only once.

```
"leetcode"

Step 1 — Count frequencies:
  l:1  e:3  t:1  c:1  o:1  d:1

Step 2 — Scan string left to right, find first with count 1:
  'l' → count=1 → FOUND!

  ┌─────────────────────────────────────┐
  │  Pass 1: Build frequency map        │
  │  Pass 2: Find first with freq == 1  │
  └─────────────────────────────────────┘
```

In [ ]:
def first_unique_char(s):
    freq = Counter(s)
    print(f"  Frequencies: {dict(freq)}")

    for i, ch in enumerate(s):
        print(f"  Checking s[{i}]='{ch}', count={freq[ch]}", end="")
        if freq[ch] == 1:
            print(" ← FIRST UNIQUE!")
            return i
        print()
    return -1


print('=== First Non-Repeating Character ===')
print('s = "leetcode"')
idx = first_unique_char("leetcode")
print(f"Result: index {idx} ('{('leetcode')[idx]}')\n")

print('s = "aabb"')
idx = first_unique_char("aabb")
print(f"Result: {idx} (no unique character)")

---

## Core Problem 4: Group Anagrams

Given a list of strings, group the ones that are anagrams of each other.

**Key insight:** if you sort the characters of an anagram, they all produce the **same key**.

```
"eat" → sorted → "aet"
"tea" → sorted → "aet"   ← same key!
"tan" → sorted → "ant"
"ate" → sorted → "aet"   ← same key!
"nat" → sorted → "ant"   ← same key!
"bat" → sorted → "abt"

Group by sorted key:
  "aet" → ["eat", "tea", "ate"]
  "ant" → ["tan", "nat"]
  "abt" → ["bat"]

  ┌──────────────────────────────────────┐
  │  groups = defaultdict(list)          │
  │  for word in words:                  │
  │      key = sorted(word) as tuple     │
  │      groups[key].append(word)        │
  └──────────────────────────────────────┘
```

In [ ]:
from collections import defaultdict


def group_anagrams(strs):
    groups = defaultdict(list)
    for word in strs:
        key = tuple(sorted(word))
        groups[key].append(word)
        print(f"  '{word}' → key={''.join(key)} → group so far: {groups[key]}")
    return list(groups.values())


print("=== Group Anagrams ===")
words = ["eat", "tea", "tan", "ate", "nat", "bat"]
print(f"Input: {words}\n")
result = group_anagrams(words)
print(f"\nGrouped: {result}")

---

## Core Problem 5: Longest Consecutive Sequence

Find the length of the longest run of consecutive numbers in an unsorted array.  
**Must be O(n)** — so sorting (O(n log n)) is off the table.

### The Trick: Use a Set

1. Put everything in a set (O(1) lookups).
2. For each number, check: is it the **start** of a sequence? (i.e., `num - 1` NOT in the set)
3. If yes, count forward: `num, num+1, num+2, ...` as long as they're in the set.

```
nums = [100, 4, 200, 1, 3, 2]
Set:  {100, 4, 200, 1, 3, 2}

Check each number:
  100: is 99 in set? NO  → start! Count: 100        → length 1
    4: is  3 in set? YES → skip (not a start)
  200: is 199 in set? NO → start! Count: 200        → length 1
    1: is  0 in set? NO  → start! Count: 1,2,3,4    → length 4  ★
    3: is  2 in set? YES → skip
    2: is  1 in set? YES → skip

Answer: 4

Why "only starts"? Without this optimization, you'd recount
overlapping sequences. With it, each number is visited at most
twice → O(n) total.
```

In [ ]:
def longest_consecutive(nums):
    num_set = set(nums)
    best = 0

    for num in num_set:
        if num - 1 not in num_set:  # only process sequence starts
            length = 1
            current = num
            while current + 1 in num_set:
                current += 1
                length += 1
            print(f"  Start at {num}: sequence = {list(range(num, num + length))} → length {length}")
            best = max(best, length)
        else:
            print(f"  {num}: {num-1} in set → skip (not a start)")

    return best


print("=== Longest Consecutive Sequence ===")
nums = [100, 4, 200, 1, 3, 2]
print(f"nums = {nums}\n")
result = longest_consecutive(nums)
print(f"\nLongest consecutive: {result}")

print("\n--- Another example ---")
nums2 = [0, 3, 7, 2, 5, 8, 4, 6, 0, 1]
print(f"nums = {nums2}\n")
result2 = longest_consecutive(nums2)
print(f"\nLongest consecutive: {result2}")

---

## Core Problem 6: Subarray Sum Equals K

Given an array of integers and a target `k`, find the **total number of contiguous subarrays** whose sum equals `k`.

This combines **prefix sums** + **hash map**. It's a pattern you MUST know.

### The Insight

```
If prefix_sum[j] - prefix_sum[i] = k,
then the subarray from index (i+1) to j sums to k.

nums = [1, 2, 3], k = 3

Index:       0    1    2
Value:       1    2    3
Prefix:  0   1    3    6
              ↑        ↑
              1   to   6  → difference = 5 (not k)
              ↑   ↑
              1   3   → difference = 2 (not k)
         ↑        ↑
         0   to   3  → difference = 3 = k!  (subarray [1,2])
         ↑   ↑
         0   1    → diff = 1 (not k)
                   ↑   ↑
                   3   6  → difference = 3 = k!  (subarray [3])

So for each prefix sum, ask:
  "How many times have I seen (current_prefix - k) before?"

  ┌──────────────────────────────────────────────┐
  │  prefix = 0, count = 0                       │
  │  prefix_counts = {0: 1}   ← empty prefix     │
  │                                              │
  │  for num in nums:                            │
  │      prefix += num                           │
  │      count += prefix_counts[prefix - k]      │
  │      prefix_counts[prefix] += 1              │
  └──────────────────────────────────────────────┘
```

### Step-by-step trace:

```
nums = [1, 2, 3], k = 3
prefix=0, prefix_counts={0:1}, count=0

num=1: prefix=1, need 1-3=-2, found 0 times. counts={0:1, 1:1}
num=2: prefix=3, need 3-3= 0, found 1 time!  counts={0:1, 1:1, 3:1}, count=1
num=3: prefix=6, need 6-3= 3, found 1 time!  counts={0:1, 1:1, 3:1, 6:1}, count=2

Answer: 2  (subarrays [1,2] and [3])
```

In [ ]:
def subarray_sum(nums, k):
    prefix = 0
    count = 0
    prefix_counts = defaultdict(int)
    prefix_counts[0] = 1  # empty prefix

    for i, num in enumerate(nums):
        prefix += num
        need = prefix - k
        found = prefix_counts[need]
        count += found
        prefix_counts[prefix] += 1
        print(f"  i={i}, num={num}, prefix={prefix}, need={need}, "
              f"found={found}, count={count}, map={dict(prefix_counts)}")

    return count


print("=== Subarray Sum Equals K ===")
nums = [1, 2, 3]
k = 3
print(f"nums = {nums}, k = {k}\n")
result = subarray_sum(nums, k)
print(f"\nSubarrays summing to {k}: {result}")

print("\n--- Harder example with negatives ---")
nums2 = [1, -1, 1, 1, -1, 1]
k2 = 1
print(f"nums = {nums2}, k = {k2}\n")
result2 = subarray_sum(nums2, k2)
print(f"\nSubarrays summing to {k2}: {result2}")

---

## Core Problem 7: Valid Anagram

Given two strings, determine if one is an anagram of the other.  
Two strings are anagrams if they contain the **exact same characters** with the **exact same frequencies**.

```
"anagram" vs "nagaram"

Counter("anagram") = {a:3, n:1, g:1, r:1, m:1}
Counter("nagaram") = {n:1, a:3, g:1, r:1, m:1}

Same? YES → valid anagram
```

In [ ]:
def is_anagram(s, t):
    return Counter(s) == Counter(t)


print("=== Valid Anagram ===")
tests = [
    ("anagram", "nagaram"),
    ("rat", "car"),
    ("listen", "silent"),
]
for s, t in tests:
    result = is_anagram(s, t)
    c1, c2 = Counter(s), Counter(t)
    print(f"  '{s}' vs '{t}': {dict(c1)} == {dict(c2)} → {result}")

---

## Core Problem 8: Intersection of Two Arrays

Find common elements between two arrays. Two approaches:

```
nums1 = [1, 2, 2, 1], nums2 = [2, 2]

Set approach (unique intersection):
  set(nums1) & set(nums2) = {1, 2} & {2} = {2}

Counter approach (with duplicates):
  Counter(nums1) = {1:2, 2:2}
  Counter(nums2) = {2:2}
  Counter & Counter  = {2:2}  → [2, 2]
  (takes min count for each element)
```

In [ ]:
print("=== Intersection of Two Arrays ===")
nums1 = [1, 2, 2, 1]
nums2 = [2, 2]
print(f"nums1 = {nums1}, nums2 = {nums2}\n")

# Unique intersection
unique = set(nums1) & set(nums2)
print(f"Unique intersection (set):  {unique}")

# Intersection with duplicates
c1, c2 = Counter(nums1), Counter(nums2)
common = c1 & c2
with_dupes = list(common.elements())
print(f"With duplicates (Counter): {with_dupes}")
print(f"  Counter(nums1) = {dict(c1)}")
print(f"  Counter(nums2) = {dict(c2)}")
print(f"  Intersection   = {dict(common)}")

---

## Core Problem 9: Contains Duplicate

In [ ]:
print("=== Contains Duplicate ===")

# One-liner: if there are fewer unique elements than total, there's a duplicate
nums = [1, 2, 3, 1]
print(f"nums = {nums}")
print(f"  One-liner: len(set(nums)) != len(nums) → {len(set(nums)) != len(nums)}")

# Explicit loop approach — stop early on first duplicate found
print("\n  Explicit loop trace:")
seen = set()
for num in nums:
    if num in seen:
        print(f"    {num} already in {seen} → DUPLICATE!")
        break
    print(f"    {num} not in {seen} → add it")
    seen.add(num)

print("\n--- No duplicates example ---")
nums2 = [1, 2, 3, 4]
print(f"nums = {nums2}")
print(f"  len(set(nums)) != len(nums) → {len(set(nums2)) != len(nums2)}")

---

## Core Problem 10: Longest Substring Without Repeating Characters

Find the length of the longest substring with all unique characters.  
This is a **sliding window + hash set** problem.

```
s = "abcabcbb"

Window slides through the string, using a SET to track current chars:

  left=0                        Set         Best
  ──────────────────────────────────────────────
  [a] b c a b c b b             {a}           1
  [a  b] c a b c b b           {a,b}          2
  [a  b  c] a b c b b         {a,b,c}         3
  [a  b  c  a] ← 'a' repeat!
      shrink: remove 'a', left=1
     [b  c  a] b c b b        {b,c,a}         3
     [b  c  a  b] ← 'b' repeat!
         shrink: remove 'b', left=2
        [c  a  b] c b b       {c,a,b}         3
  ... and so on

Answer: 3 ("abc")

  ┌────────────────────────────────────────┐
  │  Use a SET for the current window.     │
  │  Expand right: add char to set.        │
  │  If duplicate: shrink left until gone. │
  │  Track max window size.                │
  └────────────────────────────────────────┘
```

In [ ]:
def length_of_longest_substring(s):
    char_set = set()
    left = 0
    best = 0

    for right in range(len(s)):
        while s[right] in char_set:
            print(f"    '{s[right]}' duplicate! Remove '{s[left]}', left → {left+1}")
            char_set.remove(s[left])
            left += 1

        char_set.add(s[right])
        best = max(best, right - left + 1)
        window = s[left:right+1]
        print(f"  right={right} '{s[right]}': window='{window}' set={char_set} best={best}")

    return best


print("=== Longest Substring Without Repeating Characters ===")
s = "abcabcbb"
print(f's = "{s}"\n')
result = length_of_longest_substring(s)
print(f"\nLongest: {result}")

print("\n--- Another example ---")
s2 = "pwwkew"
print(f's = "{s2}"\n')
result2 = length_of_longest_substring(s2)
print(f"\nLongest: {result2}")

---

## Practice Problems

| # | Problem | LeetCode | Difficulty | Key Pattern |
|---|---------|----------|------------|-------------|
| 1 | Two Sum | [#1](https://leetcode.com/problems/two-sum/) | Easy | Complement lookup |
| 2 | Contains Duplicate | [#217](https://leetcode.com/problems/contains-duplicate/) | Easy | Set membership |
| 3 | Valid Anagram | [#242](https://leetcode.com/problems/valid-anagram/) | Easy | Counter comparison |
| 4 | First Unique Character | [#387](https://leetcode.com/problems/first-unique-character-in-a-string/) | Easy | Frequency scan |
| 5 | Intersection of Two Arrays II | [#350](https://leetcode.com/problems/intersection-of-two-arrays-ii/) | Easy | Counter intersection |
| 6 | Group Anagrams | [#49](https://leetcode.com/problems/group-anagrams/) | Medium | defaultdict + sorted key |
| 7 | Top K Frequent Elements | [#347](https://leetcode.com/problems/top-k-frequent-elements/) | Medium | Counter.most_common |
| 8 | Longest Consecutive Sequence | [#128](https://leetcode.com/problems/longest-consecutive-sequence/) | Medium | Set + sequence starts |
| 9 | Subarray Sum Equals K | [#560](https://leetcode.com/problems/subarray-sum-equals-k/) | Medium | Prefix sum + map |
| 10 | Longest Substring Without Repeating | [#3](https://leetcode.com/problems/longest-substring-without-repeating-characters/) | Medium | Sliding window + set |
| 11 | 4Sum II | [#454](https://leetcode.com/problems/4sum-ii/) | Medium | Hash map complement |
| 12 | Longest Palindrome | [#409](https://leetcode.com/problems/longest-palindrome/) | Easy | Frequency counting |

---

## Hash Map Pattern Cheat Sheet

```
HASH MAP PATTERN CHEAT SHEET:

"Find a pair/complement"       → Store seen values, check complement
"Count frequencies"            → Counter or manual dict counting
"Group by property"            → defaultdict(list) with computed key
"Check for duplicates"         → Set: add and check membership
"Longest consecutive"          → Set + find sequence starts
"Subarray sum = k"             → Prefix sum + hash map of prefix counts
"Anagram check/group"          → Sort chars as key, or Counter comparison
"Sliding window membership"    → Set to track current window contents
```

### When to reach for a hash map:

```
Ask yourself:
┌──────────────────────────────────────────────────┐
│  "Do I need to look something up quickly?"  → dict  │
│  "Do I need to check if I've seen this?"    → set   │
│  "Do I need to count things?"               → Counter│
│  "Do I need to group things?"               → defaultdict(list)│
│  "Am I doing O(n²) nested lookups?"         → dict can make it O(n)│
└──────────────────────────────────────────────────┘
```

---

**Next up: Topic 6 — Recursion & Backtracking**